In [ ]:
import pandas as pd
import hockeydata.database_session.database_session as ds
import hockeydata.database_creator.database_creator as db
import database_queries.database_query as dq

In [2]:
db_path = "../database/hockey_v17_test.db"

#connection to database is managed by class GetDatabaseSession
#which takes one parameter with the path to the existing DB
#or path where the new DB should be created in case that it does not
#already exist at a given location

session_o = ds.GetDatabaseSession(db_path=db_path)
session_o.set_up_connection()

INFO: New scrapping session started
INFO: New DB session initiated with db at ../database/hockey_v17_test.db


In [ ]:
filter = [db.Season.season.in_(["2022-2023"])]

getter = dq.ParsedDBQuery(db_session=session_o.session)



In [21]:
data_goals =  getter.get_db_query_result(query_name="goals", query_file_path='../powerbi/queries.json' )
df_goals = pd.DataFrame(data_goals)
df_goals_sum = df_goals[['player_id', 'name']].value_counts().reset_index(name='n_goals')

In [20]:
data_assits =  getter.get_db_query_result(query_name="assists", query_file_path='../powerbi/queries.json' )
df_assist = pd.DataFrame(data_assits)
df_assist_sum = df_assist[['player_id', 'name']].value_counts().reset_index(name='n_assits')

In [ ]:
df_points = pd.merge(df_goals_sum, df_assist_sum, on="player_id")
df_points["p"] = df_points['n_goals'] + df_points['n_assits']

In [ ]:
data_shifts =  getter.get_db_query_result(query_name="shifts", query_file_path='../powerbi/queries.json' )
df_shifts = pd.DataFrame(data_shifts)
df_shifts['shift_duration'] = df_shifts['shift_end'] - df_shifts["shift_start"]
df_shifts_sum = df_shifts[['id', 'shift_duration']].groupby('id').sum()

In [19]:
df_shifts_sum = df_shifts[['id', 'shift_duration']].groupby('id').sum()

In [27]:
df_points = pd.merge(df_points, df_shifts_sum, left_on="player_id", right_on="id")

In [40]:
df_points['min_per_point'] = ((df_points['shift_duration'] / df_points['p']).round(3) / 60).round(3)
df_points['min_per_goal'] = ((df_points['shift_duration'] / df_points['n_goals']).round(3) / 60).round(3)

In [41]:
df_points

,player_id,name_x,n_goals,name_y,n_assits,p,shift_duration,sec_per_point,min_per_point,min_per_goal
0,6374,Steven Stamkos,37,Steven Stamkos,39,76,66653,877.013,14.617,30.024
1,7305,Leon Draisaitl,35,Leon Draisaitl,38,73,56862,778.932,12.982,27.077
2,8192,Kirill Kaprizov,34,Kirill Kaprizov,32,66,54518,826.030,13.767,26.725
3,8127,Jason Robertson,34,Jason Robertson,13,47,49258,1048.043,17.467,24.146
4,6923,Chris Kreider,31,Chris Kreider,19,50,58094,1161.880,19.365,31.233
...,...,...,...,...,...,...,...,...,...,...
666,8224,Vitali Kravtsov,1,Vitali Kravtsov,1,2,4622,2311.000,38.517,77.033
667,8225,Logan Stanley,1,Logan Stanley,2,3,22339,7446.333,124.106,372.317
668,6088,Loui Eriksson,1,Loui Eriksson,7,8,19898,2487.250,41.454,331.633
669,8387,Alexei Toropchenko,1,Alexei Toropchenko,1,2,18949,9474.500,157.908,315.817


In [26]:
df[['player_id', 'name']].value_counts().reset_index(name='n_goals')

,player_id,name,n_goals
0,6374,Steven Stamkos,37
1,7305,Leon Draisaitl,35
2,8192,Kirill Kaprizov,34
3,8127,Jason Robertson,34
4,6923,Chris Kreider,31
...,...,...,...
731,8385,Jack Drury,1
732,6088,Loui Eriksson,1
733,8387,Alexei Toropchenko,1
734,5910,Brad Richardson,1
